# Project 1 — Bank Marketing: Predicting Term-Deposit Subscription
**Course:** DSC8307 Data Mining, Modeling and Analytics  
**Programme:** MSc Data Science and Analytics, Year 2  
**Purpose:** Develop a reproducible model to rank prospective customers for a term-deposit campaign.

## 1. Problem formulation
The target `y` is binary (`yes`/`no`). The operational objective is to identify customers with a higher probability of subscribing while controlling unnecessary calls. The positive class is expected to be the minority, so accuracy alone is not an adequate measure. We report ROC-AUC, average precision (PR-AUC), precision, recall, F1 and confusion matrices.

**Important leakage decision:** `duration` is excluded from the deployable model because call duration is not known before the call and is strongly associated with the outcome. Including it would create target leakage and an unrealistic pre-call model. The dataset documentation explicitly warns about this. `duration` is not used even as a predictor in the main experiment.

## 2. Reproducibility and imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_score, recall_score, f1_score, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay)
RANDOM_STATE = 42
DATA_PATH = "bank-additional-full.csv"
df = pd.read_csv(DATA_PATH, sep=";")
print("Shape:", df.shape)
display(df.head())

## 3. Data exploration
The supplied data contains 41,188 campaign records and 21 columns (20 predictors plus target). The target is imbalanced: approximately 88.7% `no` and 11.3% `yes`. The majority-class classifier is therefore a useful minimum benchmark, but it has zero recall for subscribers and cannot support campaign selection.

In [ ]:
print(df.info())
display(df.describe(include="all").T)
print("\\nTarget counts and proportions:")
display(pd.concat([df["y"].value_counts(), df["y"].value_counts(normalize=True).rename("proportion")], axis=1))
print("\\nUnknown-coded categorical values:")
for c in df.select_dtypes("object").columns:
    if c != "y":
        n = (df[c] == "unknown").sum()
        if n: print(f"{c}: {n} ({n/len(df):.1%})")
print("\\nExact missing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))
ax = df["y"].value_counts().plot(kind="bar", title="Target class distribution", rot=0)
ax.set_ylabel("Records"); plt.tight_layout(); plt.show()

## 4. Data preparation and justification
- Remove `duration` to prevent leakage and reflect the pre-call decision point.
- Treat `unknown` in categorical fields as an explicit category rather than deleting rows. The data dictionary describes these as unknown-coded values; deleting them could discard a large, potentially informative segment.
- `pdays=999` means the customer was not previously contacted. Convert this sentinel into a missing value and create `previously_contacted` to preserve the distinction. Median imputation is then applied within the training pipeline.
- Median-impute numeric variables, most-frequent-impute categorical variables, standardize numeric inputs for logistic regression, and one-hot encode categories. All preprocessing is fitted inside each pipeline to avoid train/test contamination.
- Use a stratified 80/20 train/test split so the minority-class proportion is preserved.

In [ ]:
X = df.drop(columns=["y", "duration"]).copy()
y = (df["y"] == "yes").astype(int)
X["previously_contacted"] = (X["pdays"] != 999).astype(int)
X["pdays"] = X["pdays"].replace(999, np.nan)
categorical = X.select_dtypes(include="object").columns.tolist()
numeric = X.select_dtypes(exclude="object").columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                       sparse_output=False))]), categorical)
])
print("Training records:", len(X_train), "Test records:", len(X_test))
print("Training positive rate:", round(y_train.mean(), 4),
      "Test positive rate:", round(y_test.mean(), 4))

## 5. Majority-class baseline
Predicting `no` for every customer gives accuracy equal to the majority share (about 88.7%), but subscriber recall and positive-class F1 are zero. This demonstrates why accuracy is misleading for this imbalanced decision problem.

In [ ]:
baseline_pred = np.zeros(len(y_test), dtype=int)
print("Majority-class accuracy:", round((baseline_pred == y_test).mean(), 4))
print(classification_report(y_test, baseline_pred, target_names=["no", "yes"], zero_division=0))
print("Confusion matrix:\\n", confusion_matrix(y_test, baseline_pred))

## 6. Model selection and ranking
Three complementary approaches are compared:
1. **Logistic regression:** interpretable linear baseline for log-odds; class weighting counters imbalance.
2. **Random forest:** captures nonlinearities and interactions with limited feature engineering; class-weighted trees reduce majority domination.
3. **Histogram gradient boosting:** sequentially fits trees to residual errors and can capture complex nonlinear relationships efficiently.

Models are ranked primarily by average precision (PR-AUC), which focuses on the minority positive class, with ROC-AUC and Brier score as supporting evidence. A single stratified holdout is used for transparent final evaluation; results should be confirmed on future campaign data before deployment.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                               random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=250,
        learning_rate=0.08, l2_regularization=1.0, random_state=RANDOM_STATE)
}
fitted, probabilities, rows = {}, {}, []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    fitted[name], probabilities[name] = pipe, p
    rows.append({"Model": name, "ROC-AUC": roc_auc_score(y_test, p),
                 "Average precision": average_precision_score(y_test, p),
                 "Brier score": brier_score_loss(y_test, p)})
results = pd.DataFrame(rows).sort_values("Average precision", ascending=False)
display(results)

In [ ]:
plt.figure(figsize=(7,5))
for name, p in probabilities.items():
    PrecisionRecallDisplay.from_predictions(y_test, p, name=name, ax=plt.gca())
plt.title("Precision–recall curves"); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,5))
for name, p in probabilities.items():
    RocCurveDisplay.from_predictions(y_test, p, name=name, ax=plt.gca())
plt.plot([0,1],[0,1],"k--",alpha=.5); plt.title("ROC curves")
plt.tight_layout(); plt.show()

## 7. Decision threshold and operational trade-off
A default threshold of 0.50 is not automatically appropriate. A false positive represents an unnecessary call; a false negative represents a potential subscriber not contacted. For an illustrative campaign policy, assign a missed subscriber three times the cost of an unnecessary call (false-negative cost = 3, false-positive cost = 1). This is a stated planning assumption, not a measured financial fact. Compare thresholds on the held-out set and select the threshold minimizing this weighted error.

The illustrative validation supports a threshold near **0.20** for the histogram gradient boosting model under this cost ratio. At lower thresholds more potential subscribers are reached, but the call list includes more non-subscribers. The campaign should replace the assumed cost ratio with actual call cost, expected deposit contribution and contact capacity before operational use.

In [ ]:
best_name = results.iloc[0]["Model"]
best_p = probabilities[best_name]
threshold_rows = []
for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred = (best_p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    threshold_rows.append({"threshold": threshold, "precision": precision_score(y_test,pred,zero_division=0),
        "recall": recall_score(y_test,pred), "F1": f1_score(y_test,pred),
        "false_positive_calls": fp, "missed_subscribers": fn,
        "weighted_cost_per_record": (fp + 3*fn)/len(y_test)})
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
chosen_threshold = float(threshold_table.loc[threshold_table["weighted_cost_per_record"].idxmin(), "threshold"])
print("Cost-minimising threshold under FP=1, FN=3:", chosen_threshold)
final_pred = (best_p >= chosen_threshold).astype(int)
print(classification_report(y_test, final_pred, target_names=["no","yes"], zero_division=0))
print("Confusion matrix:\\n", confusion_matrix(y_test, final_pred))

## 8. Findings, limitations and recommendation
**Evidence from the supplied split (random state 42):** histogram gradient boosting achieved the strongest average precision (approximately 0.495) and ROC-AUC (approximately 0.814), narrowly/clearly ahead of the other tested methods on these metrics. Random forest was second by average precision (approximately 0.481), while logistic regression was third (approximately 0.460). The majority baseline's accuracy is high only because `no` dominates.

**Recommendation:** use the histogram gradient boosting model to rank customers and pilot a threshold around 0.20 only if the assumed cost ratio is accepted. A ranked top-N call list may be preferable when agent capacity is fixed. Calibrate probabilities and test thresholds using campaign economics and a temporally separated validation set before deployment.

**Limitations:** records are observational and from one institution/campaign context; class prevalence and customer behavior may change. The random split may not represent future-time performance. Macro-level socioeconomic variables and campaign features may be correlated. The model estimates association, not the causal effect of calling a customer. Fairness, privacy, consent, contact-frequency limits and opt-out rules must be considered. Do not treat model scores as permission to contact customers.